# 0 — Exact units and quantities

Coordinates in this library are *identities*, not approximations: slicing,
lattice membership, and guard algebra all rely on exact equality. So the unit
system is **exactness-first**: magnitudes are `fractions.Fraction`, unit
conversions are exact rationals, and floats are rejected at every boundary —
because `0.1` the float is not `1/10`.

The UX is pint-flavored (`u.um`, `q("0.75 um")`), but the system is owned by
the library and deliberately minimal: a labeling algebra, not a physics
package.

In [1]:
from fractions import Fraction

import numpy as np
from nbhelp import show  # also puts tensorlib on sys.path
from tensorlib import Tensor, q, u

## Constructing quantities

Strings parse exactly (decimals, exponents, and `p/q` fractions); ints and
`Fraction`s multiply units directly.

In [2]:
q("0.75 um"), q("3/4 um"), 3 * u.mm, Fraction(3, 4) * u.um, q("1e-3 m")

(0.75 um, 0.75 um, 3 mm, 0.75 um, 0.001 m)

In [3]:
q("0.1 um").magnitude  # exactly 1/10 — parsed as a decimal, never a float

Fraction(1, 10)

## t-strings (PEP 750)

A t-string interpolation carries the *value*, not its float repr — so ints
and Fractions pass through exactly.

In [4]:
val = Fraction(1, 3)
q(t"{val} m")

1/3 m

## Floats are rejected everywhere

There is no silent 0.1 → 3602879701896397/36028797018963968 footgun; the
error tells you the exact alternatives.

In [5]:
try:
    0.1 * u.um
except TypeError as e:
    print("rejected:", e)

rejected: magnitude 0.1: floats are inexact — write the value as a string ('0.1'), a Fraction, or a t-string interpolating an int/Fraction


## Arithmetic is exact across units

Same-dimension quantities add and compare through exact conversion; the
ratio of two like quantities is a plain `Fraction`.

In [6]:
q("1 mm") + q("250 um"), q("999 um") < q("1 mm"), q("1 mm") == q("1000 um")

(1.25 mm, True, True)

In [7]:
r = q("1 mm") / q("250 um")
r, type(r).__name__

(Fraction(4, 1), 'Fraction')

In [8]:
q("3 m") / q("2 s")  # different dims: a compound unit, still exact

1.5 m/s

## Defining units

`define` adds an exact alias or a brand-new base dimension — this is the
whole extension mechanism.

In [9]:
u.define("px", dim="pixel")
u.define("tick", "1/60 s")
q("1920 px"), q("120 tick").to(u.s)

(1920 px, 2 s)

## Where this is going

These quantities become *coordinate labels* on tensor dims — charts, next
notebook. A taste:

In [10]:
arr = np.arange(5, dtype=np.int64) * 100
stage = Tensor.from_numpy(arr, ("x",)).with_charts(x=("0 um", "0.25 um"))
show(stage, "a charted dim")
stage.item(x=q("0.5 um"))

-- a charted dim
Tensor[int64] on Buffer(40B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  x            8      0      5      5  pos[x: 0 um step 0.25 um]
  numel=5  footprint=(0, 40)  injectivity=injective


np.int64(200)